# Phase I: Extract frames containing the other robot using ground truth (Vicon) from ROS2 files

This notebook contains the data-generation workflow. It synchronizes Vicon ground-truth poses to every depth image timestamp, projects the other robot into the observer camera, keeps every visible frame, extracts the original compressed-depth PNG, and writes an annotated bounding-box preview.

In [ ]:
# Install dependencies in Colab; locally use requirements.txt.
import sys
if "google.colab" in sys.modules:
    %pip install -q pillow pandas matplotlib numpy opencv-python-headless

## Locate the kit and configure bags

For Colab, upload this kit and the three SQLite ROS 2 bag files to Drive. Change the paths below if your Drive layout differs. The precomputed CSVs and 12 example extractions let every later inspection cell run even when the large bags are not uploaded.

In [ ]:
from pathlib import Path
import subprocess, sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

candidates = [Path.cwd(), Path.cwd().parent,
              Path('/content/drive/MyDrive/arp-prp-vicon-frame-extraction')]
ROOT = next((p for p in candidates if (p / 'snippets/phase1_depth').exists()), None)
assert ROOT is not None, 'Set ROOT to the arp-prp-vicon-frame-extraction repository directory'
DATA = ROOT / 'snippets/phase1_depth'

# Edit these three paths for a full extraction.
BAG_ROOT = ROOT / 'bags'
FUSION_DB = BAG_ROOT / 'fusion_bag_0.db3'  # /robot_a/ground_truth and /robot_b/ground_truth
TB1_DB = BAG_ROOT / 'tb1_exp_2026_04_24-12_13_33_0.db3'
TB2_DB = BAG_ROOT / 'tb2_exp_2026_04_24-12_13_28_0.db3'
OUTPUT = ROOT / 'outputs/visible_frames'
print('Kit:', ROOT)

## Projection used in the experiment

For each image timestamp, the nearest TB1 and TB2 Vicon poses are accepted only when both are within 20 ms. The target displacement is transformed into the observer camera frame. A frame is visible when the target is in front, its projected horizontal centre lies inside the image, and range is at most 6 m.

The box equations are `u = cx - fx*y/x`, `width = fx*robot_width/x`, and `height = fy*robot_height/x`. Vertical placement uses each camera's calibrated principal point (`cy`), scaled to the extracted resolution. The extraction CSV records the final clipped corner coordinates `(x0, y0, x1, y1)`.

In [ ]:
# Exact frozen parameters
PROJECTION_ARGS = [
    '--robot-width-m', '0.36', '--robot-height-m', '0.35',
    '--tb1-camera-x-m', '0.0', '--tb1-camera-y-m', '0.0',
    '--tb1-camera-yaw-offset-deg', '0.0',
    '--tb2-camera-x-m', '0.0', '--tb2-camera-y-m', '0.0',
    '--tb2-camera-yaw-offset-deg', '0.0',
    '--max-range-m', '6.0', '--max-sync-ms', '20.0',
]

TB1_TOPIC = '/tb1/oakd/stereo/image_raw/compressedDepth'
TB2_TOPIC = '/tb2/oakd/stereo/image_raw/compressedDepth'

# Intrinsics before scaling to the decoded depth-image resolution:
# TB1: 1280x720, fx=fy=1026.157958984375, cx=649.09716796875, cy=369.48046875
# TB2: 1280x720, fx=fy=1032.688232421875, cx=638.600830078125, cy=362.9078369140625

## Run the complete extraction

Set `RUN_FULL_EXTRACTION = True` after placing the bags at the configured paths. The first script reads SQLite rosbag messages directly, parses Vicon odometry CDR, and writes visibility CSVs. The second uses `--all`, so it extracts every visible timestamp, renders the improved metric-depth preview, and saves final box corners. Both scripts are included under `src/`.

In [ ]:
RUN_FULL_EXTRACTION = False

if RUN_FULL_EXTRACTION:
    required = [FUSION_DB, TB1_DB, TB2_DB]
    missing = [str(path) for path in required if not path.exists()]
    assert not missing, f'Missing bag files: {missing}'
    visibility_dir = OUTPUT / 'visibility_csv'
    visibility_dir.mkdir(parents=True, exist_ok=True)

    subprocess.run([
        sys.executable, str(ROOT / 'src/camera_visibility_scan.py'),
        '--fusion-db', str(FUSION_DB), '--tb1-db', str(TB1_DB), '--tb2-db', str(TB2_DB),
        '--output-dir', str(visibility_dir), *PROJECTION_ARGS,
    ], check=True)

    jobs = [
        (TB1_DB, TB1_TOPIC, visibility_dir / 'tb1_camera_sees_tb2.csv',
         'tb1_sees_tb2', OUTPUT / 'tb1_camera_sees_tb2', 369.48046875 / 720.0),
        (TB2_DB, TB2_TOPIC, visibility_dir / 'tb2_camera_sees_tb1.csv',
         'tb2_sees_tb1', OUTPUT / 'tb2_camera_sees_tb1', 362.9078369140625 / 720.0),
    ]
    for database, topic, visibility_csv, label, output_dir, center_v_fraction in jobs:
        subprocess.run([
            sys.executable, str(ROOT / 'src/extract_visibility_frames.py'),
            '--db', str(database), '--topic', topic, '--csv', str(visibility_csv),
            '--label', label, '--output-dir', str(output_dir), '--all',
            '--center-v-fraction', str(center_v_fraction),
        ], check=True)
    print('Extracted all visible frames under', OUTPUT)
else:
    print('Dry run: set RUN_FULL_EXTRACTION=True when the three bag files are available.')

## Verify the already computed visibility result

These are the complete CSV outputs from the original run, not a sampled table. Together they identify all 2,105 visible image timestamps.

In [ ]:
import pandas as pd

tb1_sees_tb2 = pd.read_csv(DATA / 'tb1_camera_sees_tb2.csv')
tb2_sees_tb1 = pd.read_csv(DATA / 'tb2_camera_sees_tb1.csv')
summary = pd.DataFrame({
    'observer': ['TB1 camera', 'TB2 camera'],
    'target': ['TB2', 'TB1'],
    'checked_frames': [len(tb1_sees_tb2), len(tb2_sees_tb1)],
    'visible_frames': [int(tb1_sees_tb2.visible.sum()), int(tb2_sees_tb1.visible.sum())],
})
assert summary.visible_frames.sum() == 2105
summary

In [ ]:
visible = pd.concat([
    tb1_sees_tb2.query('visible == 1').assign(observer='tb1', target='tb2'),
    tb2_sees_tb1.query('visible == 1').assign(observer='tb2', target='tb1'),
], ignore_index=True)
visible[['observer','target','stamp_ns','range_m','bearing_deg','u_px',
         'bbox_w_px','bbox_h_px','sync_dt_ms']].head(10)

## Inspect extracted frames and boxes

The kit includes 12 positive raw frames from the complete extraction. They are rendered with one fixed metric Turbo scale: warm colors are near, cool colors are far, and missing depth is black. White boxes use calibrated `cy` and exact projected dimensions without artificial minimum-size enlargement.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
sys.path.insert(0, str(ROOT / 'src'))
from extract_visibility_frames import annotate, colorize_depth

manifest = pd.read_csv(DATA / 'manifest.csv')
fig, axes = plt.subplots(3, 4, figsize=(15, 9), constrained_layout=True)
for ax, row in zip(axes.flat, manifest.itertuples(index=False)):
    png_bytes = (DATA / row.raw_path).read_bytes()
    preview = colorize_depth(png_bytes, min_depth_m=.35, max_depth_m=6.0)
    center_v_fraction = (369.48046875 if row.observer_robot == 'tb1' else 362.9078369140625) / 720.0
    metadata = row._asdict()
    metadata['time_s_from_start'] = 0.0
    annotated, box = annotate(preview, metadata, f'{row.observer_robot}_sees_{row.target_robot}',
                              center_v_fraction)
    ax.imshow(annotated)
    ax.set_title(f'{row.observer_robot} sees {row.target_robot}: {row.range_m:.2f} m')
    ax.axis('off')
plt.show()

## Outputs

Each observer directory contains matching `*_raw.png` and `*_annotated.png` files plus `extracted_frames.csv`. That CSV records timestamps, relative pose, range, bearing, synchronization error, and final `bbox_x0_px`, `bbox_y0_px`, `bbox_x1_px`, `bbox_y1_px` values. For another dataset, change bag paths, topic names, camera intrinsics, camera extrinsics, principal-point fractions, and physical robot dimensions.